# LLM Insight Generation Pipeline

Take transcripts from the ingestion pipeline and pass them into an LLM to extract **claims**, **narratives**, and **trends**.

Additional notes --> things to implement:
- Write a Provider interface (LLMProvider.generateJSON(schema, text)).
-Force structured output (JSON) and validate it (Zod/Pydantic).

Implement 2 providers:
- OllamaProvider
- BedrockProvider
- Put provider choice behind env vars:
    - ``LLM_PROVIDER=ollama | bedrock``
    - ``LLM_MODEL=llama3 | anthropic.claude... | ...``

```
┌─────────────────────────────────────────┐
│    YouTube Transcripts (from earlier)   │
└──────────────────┬──────────────────────┘
                   │
                   ▼
        ┌──────────────────────┐
        │  LLM Insight Pipeline│
        └──────────┬───────────┘
                   │
         ┌─────────▼─────────┐
         │   Check Env Var   │
         │  LLM_PROVIDER?    │
         └─────────┬─────────┘
                   │
        ┌──────────┴──────────┐
        │                     │
        ▼                     ▼
   ┌─────────────┐       ┌──────────────┐
   │   Ollama    │       │   Bedrock    │
   │  (Local)    │       │  (AWS Cloud) │
   └──────┬──────┘       └──────┬───────┘
          │                     │
          └──────────┬──────────┘
                     │
                     ▼
        ┌──────────────────────┐
        │   Extract & Format   │
        │ • Claims             │
        │ • Narratives         │
        │ • Trends             │
        │ (As structured JSON) │
        └──────────────────────┘
```

In [1]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Protocol
from abc import ABC, abstractmethod

# Using Dataclasses --> less boilerplate bs
@dataclass
class TranscriptRecord:
    """
    This data model represents one individual video transcript, with the following attributes:
    - video_id: unique identifier for each video, from YouTube's API
    - cleaned_txt_path: path to cleaned .txt
    - raw_json_path: raw JSON transcript (includes timestamps, segments for source-tracking)
    """

    video_id: str
    cleaned_txt_path: Path
    raw_json_path: Optional[Path] = None

@dataclass
class GeneratedInsights:
    """
    This data model houses. LLM-generated outputs + video metadata for storage
    """
    
    video_id: str
    claims: List[Dict[str, Any]]
    narratives: List[Dict[str, Any]]
    model: str
    provider: str
    source_cleaned_txt: str
    source_raw_json: Optional[str]
    chunk_count: int



class LLMProvider(ABC):
    """
    Base interface structure --> Any LLM provider in this pipeline must use this class. 
    - Allows for faster migration/transition from one LLM to another
        --> ie. Ollama (local) to AWS Bedrock/Kendra

    Each provider must define 2 attributes:
    - provider = (OpenAI, Anthropic, Google, etc.)
    - model = specific LLM configuration we're working with
    """
    def __init__(self, provider: str, model:str):
        self.provider = provider
        self.model = model

    @abstractmethod
    def generate_response(self, *, system: str, user_prompt: str) -> str:
        """
        Takes a system message & user prompt for input and returns raw JSON formatted responses.
        - Interface doesn't fully outline function behavior becuase response generation varies from provider to provider.
            --> HTTP Call?
            --> boto3 SDK Call?
        """
        pass

## 1. Load & Build Transcripts

Read cleaned transcripts from storage. Support single or batch. Keep source metadata for traceability.

In [2]:
import os
import sys
from pathlib import Path
from typing import Union

TRANSCRIPT_ROOT_DIR = "data/transcripts"

def load_cleaned_transcript(fp: Union[str, Path]) -> str:
    """
    Load a single cleaned transcript file and return its text content.
    """
    file_path = Path(fp)
    if not file_path.exists():
        raise FileNotFoundError(f"Transcript file not found: {file_path}")
    if file_path.suffix != ".txt":
        raise ValueError(f"File is not formatted as a .txt file: {file_path}")
    return file_path.read_text(encoding="utf-8")


def transcript_record_from_path(fp: Union[str, Path]) -> TranscriptRecord:
    """
    Build a TranscriptRecord from a single cleaned transcript file path.
    Derives video_id from the file stem (e.g. gpzDxm7qflY).
    """
    path = Path(fp).resolve()
    video_id = path.stem
    return TranscriptRecord(
        video_id=video_id,
        cleaned_txt_path=path,
        raw_json_path=None,
    )

## 2. Prepare for Analysis

Format transcript for the model. Handle length limits (chunk, truncate, or summarize). Handle edge cases.

In [3]:
import json
from typing import Any, Dict, List

def chunk_text(text: str, max_chars: int = 12000) -> List[str]:
    """
    Splits long transcript text into chunks to fit model context limits.
    Uses paragraph boundaries when possible.
    """
    paragraphs = text.split("\n\n")
    chunks: List[str] = []
    buffer: List[str] = []
    current_len = 0

    for para in paragraphs:
        para_len = len(para) + 2  # +2 for "\n\n"
        if current_len + para_len > max_chars and buffer:
            chunks.append("\n\n".join(buffer))
            buffer = [para]
            current_len = len(para)
        else:
            buffer.append(para)
            current_len += para_len

    if buffer:
        chunks.append("\n\n".join(buffer))
    return chunks if chunks else [text]


def validate_json_output(text: str) -> Dict[str, Any]:
    """
    Parses and sanity-checks the model output as JSON.
    Requires "claims" key (list). Narratives optional for this mini workflow.
    """
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}") from e

    if not isinstance(parsed, dict):
        raise ValueError("Model output must be a JSON object (dict)")

    if "claims" not in parsed:
        raise ValueError("Model output must contain a 'claims' key")

    if not isinstance(parsed["claims"], list):
        raise ValueError("'claims' must be a list")

    return parsed

## 3. Define Output Structure

Specify what **claims**, **narratives**, and **trends** look like. Use this to instruct the model and parse responses.

In [4]:
SYSTEM = "You extract factual claims from the transcript. Return ONLY valid JSON."
SCHEMA_HINT = """
{
  "claims": [{"text": "string", "confidence": 0.0}]
}
""".strip()

def build_user_prompt(transcript_chunk: str) -> str:
    """
    Builds the user prompt for the LLM, including the JSON shape we expect.
    """
    return f"""Extract the main factual claims from the following transcript.

Expected JSON output format (return ONLY valid JSON, no other text):
{SCHEMA_HINT}

Transcript:
---
{transcript_chunk}
---

Return your response as a single JSON object with a "claims" array. Each claim should have "text" (the claim) and "confidence" (0.0 to 1.0)."""


def extract_insights_for_record(record: TranscriptRecord, provider: LLMProvider, *, max_chars: int = 12000, retries: int = 2) -> GeneratedInsights:
    """
    Loads the cleaned transcript, chunks it, runs the provider on each chunk,
    merges results, and returns a single GeneratedInsights.
    """
    text = load_cleaned_transcript(record.cleaned_txt_path)
    chunks = chunk_text(text, max_chars=max_chars)
    all_claims: List[Dict[str, Any]] = []

    for chunk in chunks:
        user_prompt = build_user_prompt(chunk)
        last_error = None
        for attempt in range(retries + 1):
            try:
                raw = provider.generate_response(system=SYSTEM, user_prompt=user_prompt)
                parsed = validate_json_output(raw)
                all_claims.extend(parsed.get("claims", []))
                break
            except Exception as e:
                last_error = e
                if attempt == retries:
                    raise last_error from last_error

    # Deduplicate by claim text
    seen: set[str] = set()
    unique_claims: List[Dict[str, Any]] = []
    for c in all_claims:
        t = c.get("text", "")
        if t and t not in seen:
            seen.add(t)
            unique_claims.append(c)

    return GeneratedInsights(
        video_id=record.video_id,
        claims=unique_claims,
        narratives=[],
        model=provider.model,
        provider=provider.provider,
        source_cleaned_txt=str(record.cleaned_txt_path),
        source_raw_json=str(record.raw_json_path) if record.raw_json_path else None,
        chunk_count=len(chunks),
    )

## 4. Run Model

Send transcript + instructions to the model. Retrieve the response. Handle failures and limits.

Separate implementation for:
- Ollama
- AWS Bedrock

In [5]:
import requests

class OllamaProvider(LLMProvider):
    """
    Calls local Ollama using an OpenAI-compatible endpoint.
    """
    name = "ollama"

    def __init__(self, model: str = "llama3", base_url: str = "http://localhost:11434/v1"):
        super().__init__(provider="ollama", model=model)
        self.base_url = base_url.rstrip("/")

    def generate_response(self, *, system: str, user_prompt: str) -> str:
        url = f"{self.base_url}/chat/completions"
        model_to_use = self.model
        payload = {
            "model": model_to_use,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": 0.3,
            "stream": False,
        }
        resp = requests.post(url, json=payload, timeout=120)
        if not resp.ok:
            err_msg = resp.text
            try:
                err = resp.json()
                if isinstance(err.get("error"), dict):
                    err_msg = err["error"].get("message", err_msg)
                elif isinstance(err.get("error"), str):
                    err_msg = err["error"]
            except Exception:
                pass
            hint = ""
            if "not found" in err_msg.lower():
                hint = " Run `ollama pull llama3` (or another model) to download a model first."
            raise RuntimeError(f"Ollama API error ({resp.status_code}): {err_msg}.{hint}") from None
        data = resp.json()
        content = data["choices"][0]["message"]["content"]
        return content

In [6]:
# End-to-end test: extract claims from gpzDxm7qflY.txt using Ollama
import os
from pathlib import Path

# Resolve path: try relative to cwd (backend/ or pipelines/)
for base in [Path.cwd(), Path.cwd().parent]:
    candidate = base / "data" / "transcripts" / "cleaned" / "gpzDxm7qflY.txt"
    if candidate.exists():
        TRANSCRIPT_PATH = candidate
        break
else:
    TRANSCRIPT_PATH = Path("data/transcripts/cleaned/gpzDxm7qflY.txt").resolve()

record = transcript_record_from_path(TRANSCRIPT_PATH)
model = os.environ.get("LLM_MODEL", "llama3")
provider = OllamaProvider(model=model)

insights = extract_insights_for_record(record, provider)

print(f"Video ID: {insights.video_id}")
print(f"Chunks processed: {insights.chunk_count}")
print(f"Claims extracted: {len(insights.claims)}")
print("\nClaims:")
for i, claim in enumerate(insights.claims, 1):
    text = claim.get("text", "")
    conf = claim.get("confidence", "N/A")
    print(f"  {i}. {text} (confidence: {conf})")

Video ID: gpzDxm7qflY
Chunks processed: 1
Claims extracted: 5

Claims:
  1. Health isn't about obsession. (confidence: 1.0)
  2. Health isn't about judgment. (confidence: 1.0)
  3. Health is about creating sustainable habits that can serve you for a lifetime. (confidence: 1.0)
  4. Health is about giving yourself space to enjoy your life in the process. (confidence: 1.0)
  5. Health is about building the right foundation so that you can have a bag of potato chips every couple of weeks or a pizza every month with friends. (confidence: 1.0)


# **TO BE IMPLEMENTED** --> Ignore beyond this point

In [7]:
class BedrockProvider(LLMProvider):
    """
    Calls Amazon Bedrock using the Converse API.
    """
    name = "bedrock"

    def __init__(self, model: str, region: str = "us-east-1"):
        super().__init__(provider="bedrock", model=model)
        self.region = region
        # PSEUDOCODE: init bedrock-runtime client for region

    def generate_response(self, *, system: str, user_prompt: str) -> str:
        # PSEUDOCODE:
        # 1. Call converse API: modelId, system message, user message, inference config (temp, maxTokens)
        # 2. Extract text from response (output.message.content[0].text)
        # 3. Return raw string (expected to be JSON)
        pass

## 5. Extract & Validate Insights

Parse the model response into structured insights. Validate against the defined structure. Link back to source.

## 6. Persist Insights

Write insights to storage. Use a consistent format. Handle re-runs (idempotency or versioning).

## Flow

Load → Prepare → Define structure → Run model → Extract & validate → Persist

In [8]:
def build_transcript_records(cleaned_dir: str, raw_json_dir: str | None) -> list[TranscriptRecord]:
    """
    Scans cleaned_dir (and optionally raw_json_dir) and builds TranscriptRecord for each transcript.
    """
    # PSEUDOCODE:
    # 1. List .txt files in cleaned_dir
    # 2. For each file: derive video_id (e.g. from filename), resolve cleaned_txt_path
    # 3. If raw_json_dir: try to find matching raw JSON by video_id
    # 4. Yield/return list of TranscriptRecord
    pass

def run_batch(cleaned_dir: str, raw_json_dir: str | None, provider: LLMProvider):
    """
    Builds records from your folder, runs extraction per transcript,
    and prints a quick summary per file.
    """
    # PSEUDOCODE:
    # 1. records = build_transcript_records(cleaned_dir, raw_json_dir)
    # 2. For each record: call extract_insights_for_record(record, provider)
    # 3. On success: print summary (video_id, claim count, narrative count, chunks)
    # 4. On failure: print error, continue to next
    # 5. (Optional) Persist each GeneratedInsights to storage

    pass